Здесь реализовал простую двуслойную нейронную сеть FFNN, которая решает задачу XOR, а также реализовал свой SGD оптимизатор.

In [42]:
import torch
import numpy as np

In [ ]:
class Linear:
    def __init__(self, in_f, out_f):
        self.weight = torch.tensor(np.random.randn(in_f, out_f), dtype=torch.float, requires_grad=True)
        self.bias = torch.tensor(np.random.randn(out_f), dtype=torch.float, requires_grad=True)

    def __call__(self, x):
        self.linear = x @ self.weight + self.bias
        return self.linear
    
    def parameters(self):
        yield self.weight
        yield self.bias
    
class Tanh:
    def __call__(self, x):
        self.tanh = torch.tanh(x)
        return self.tanh
    
class Sigmoid:
    def __call__(self, x):
        self.sigmoid = torch.sigmoid(x)
        return self.sigmoid
    
class SGD: # По факту, когда мы тренируемся на всем наборе, а не на батчах, то это GD! (.grad для каждого веса = среднее градиентов этого веса по каждому наблюдению)
    def __init__(self, model_parameters, lr=0.1):
        # очень важно поставить list, т.к. генератор заканчивается после первого обучения, и весь процесс встает
        self.params = list(model_parameters) 
        self.lr = lr

    def step(self):
        with torch.no_grad():
            for p in self.params:
                if p.grad is not None:
                    p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


In [44]:
class Net:
    def __init__(self, in_f, out_f):
        self.f1 = Linear(in_f, 2)
        self.act1 = Sigmoid()

        self.f2 = Linear(2, out_f)
        self.act2 = Sigmoid()

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        x = self.act1(self.f1(x))
        x = self.act2(self.f2(x))
        return x
    
    def parameters(self):
        yield from self.f1.parameters()
        yield from self.f2.parameters()

In [45]:
net = Net(2, 1)
optimizer = SGD(net.parameters(), lr=1)
criterion = torch.nn.BCELoss()

In [46]:
X = torch.tensor([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1],], dtype=torch.float)
y = torch.tensor([0, 1, 1, 0], dtype=torch.float).view(-1, 1)

In [47]:
def train(X, y, model, criterion, optimizer, epoch=10):
    for e in range(epoch):
        y_pred = model(X)

        loss = criterion(y_pred, y)
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        print(f'{e:<3}:: {loss:.4f}')

In [48]:
train(X, y, net, criterion, optimizer, 1000)

0  :: 0.7027
1  :: 0.6989
2  :: 0.6975
3  :: 0.6969
4  :: 0.6966
5  :: 0.6964
6  :: 0.6963
7  :: 0.6961
8  :: 0.6960
9  :: 0.6959
10 :: 0.6958
11 :: 0.6957
12 :: 0.6956
13 :: 0.6955
14 :: 0.6954
15 :: 0.6953
16 :: 0.6952
17 :: 0.6951
18 :: 0.6950
19 :: 0.6949
20 :: 0.6949
21 :: 0.6948
22 :: 0.6947
23 :: 0.6947
24 :: 0.6946
25 :: 0.6945
26 :: 0.6945
27 :: 0.6944
28 :: 0.6944
29 :: 0.6943
30 :: 0.6943
31 :: 0.6942
32 :: 0.6942
33 :: 0.6941
34 :: 0.6941
35 :: 0.6941
36 :: 0.6940
37 :: 0.6940
38 :: 0.6939
39 :: 0.6939
40 :: 0.6939
41 :: 0.6938
42 :: 0.6938
43 :: 0.6938
44 :: 0.6937
45 :: 0.6937
46 :: 0.6937
47 :: 0.6936
48 :: 0.6936
49 :: 0.6936
50 :: 0.6936
51 :: 0.6935
52 :: 0.6935
53 :: 0.6935
54 :: 0.6935
55 :: 0.6934
56 :: 0.6934
57 :: 0.6934
58 :: 0.6934
59 :: 0.6933
60 :: 0.6933
61 :: 0.6933
62 :: 0.6933
63 :: 0.6933
64 :: 0.6932
65 :: 0.6932
66 :: 0.6932
67 :: 0.6932
68 :: 0.6931
69 :: 0.6931
70 :: 0.6931
71 :: 0.6931
72 :: 0.6931
73 :: 0.6930
74 :: 0.6930
75 :: 0.6930
76 :: 0.6930

In [49]:
net(X)

tensor([[0.0174],
        [0.9776],
        [0.9835],
        [0.0148]], grad_fn=<SigmoidBackward0>)

In [50]:
model = torch.nn.Sequential(
    torch.nn.Linear(2, 2),
    torch.nn.Sigmoid(),
    torch.nn.Linear(2, 1),
    torch.nn.Sigmoid(),
)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1)

In [51]:
train(X, y, model, criterion, optimizer, 1000)

0  :: 0.6951
1  :: 0.6938
2  :: 0.6933
3  :: 0.6931
4  :: 0.6930
5  :: 0.6930
6  :: 0.6930
7  :: 0.6930
8  :: 0.6930
9  :: 0.6930
10 :: 0.6930
11 :: 0.6930
12 :: 0.6930
13 :: 0.6930
14 :: 0.6930
15 :: 0.6930
16 :: 0.6930
17 :: 0.6930
18 :: 0.6930
19 :: 0.6930
20 :: 0.6930
21 :: 0.6930
22 :: 0.6930
23 :: 0.6930
24 :: 0.6930
25 :: 0.6930
26 :: 0.6930
27 :: 0.6930
28 :: 0.6930
29 :: 0.6930
30 :: 0.6929
31 :: 0.6929
32 :: 0.6929
33 :: 0.6929
34 :: 0.6929
35 :: 0.6929
36 :: 0.6929
37 :: 0.6929
38 :: 0.6929
39 :: 0.6929
40 :: 0.6929
41 :: 0.6929
42 :: 0.6929
43 :: 0.6929
44 :: 0.6929
45 :: 0.6929
46 :: 0.6929
47 :: 0.6929
48 :: 0.6929
49 :: 0.6929
50 :: 0.6929
51 :: 0.6929
52 :: 0.6929
53 :: 0.6929
54 :: 0.6929
55 :: 0.6929
56 :: 0.6929
57 :: 0.6929
58 :: 0.6929
59 :: 0.6929
60 :: 0.6929
61 :: 0.6929
62 :: 0.6929
63 :: 0.6929
64 :: 0.6929
65 :: 0.6929
66 :: 0.6929
67 :: 0.6929
68 :: 0.6929
69 :: 0.6929
70 :: 0.6929
71 :: 0.6929
72 :: 0.6929
73 :: 0.6929
74 :: 0.6928
75 :: 0.6928
76 :: 0.6928

In [52]:
model(X)

tensor([[0.0226],
        [0.9714],
        [0.9784],
        [0.0192]], grad_fn=<SigmoidBackward0>)